In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_aws import ChatBedrockConverse
import re
llm_nova = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1",
    temperature=0.5,
    max_tokens=200
)


In [2]:
# ---------------------------------------------------
# State Definition
# ---------------------------------------------------
class AgentState(TypedDict):
    topic: str
    draft: str
    critique: str
    score: int
    iteration: int


In [3]:
# Nodes
# ---------------------------------------------------
def generate(state: AgentState):
    response = llm_nova.invoke(
        f"Explain {state['topic']} clearly in 250 words."
    )
    return {
        "draft": response.content,
        "iteration": 0
    }
def critique(state: AgentState):
    response = llm_nova.invoke(
        f"""
        Evaluate this explanation from 0-10 on:
        clarity, accuracy, depth.
        Return:
        Overall Score: <number>
        Explanation:
        {state['draft']}
        """
    )
    text = response.content
    match = re.search(r"(\d+)", text)
    score = int(match.group()) if match else 0
    return {
        "critique": text,
        "score": score,
        "iteration": state["iteration"] + 1
    }
def revise(state: AgentState):
    response = llm_nova.invoke(
        f"""
        Improve the explanation using the critique:
        Explanation:
        {state['draft']}
        Critique:
        {state['critique']}
        """
    )
    return {"draft": response.content}


In [4]:
# Conditional Logic
# ---------------------------------------------------
MAX_ITERATIONS = 3
THRESHOLD = 8
def should_continue(state: AgentState):
    if state["score"] >= THRESHOLD:
        return "end"
    if state["iteration"] >= MAX_ITERATIONS:
        return "end"
    return "revise"


In [5]:
# Graph Construction
# ---------------------------------------------------
builder = StateGraph(AgentState)
builder.add_node("generate", generate)
builder.add_node("critique", critique)
builder.add_node("revise", revise)
builder.set_entry_point("generate")
builder.add_edge("generate", "critique")
builder.add_conditional_edges(
    "critique",
    should_continue,
    {
        "revise": "revise",
        "end": END
    }
)
builder.add_edge("revise", "critique")
graph = builder.compile()


In [7]:
from IPython.display import display, Markdown
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)
display(Markdown(f"```mermaid\n{mermaid}\n```"))


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	critique(critique)
	revise(revise)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	critique -. &nbsp;end&nbsp; .-> __end__;
	critique -.-> revise;
	generate --> critique;
	revise --> critique;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	critique(critique)
	revise(revise)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	critique -. &nbsp;end&nbsp; .-> __end__;
	critique -.-> revise;
	generate --> critique;
	revise --> critique;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
# Run
# ---------------------------------------------------
result = graph.invoke({"topic": "Retrieval-Augmented Generation"})
print("\nFinal Draft:\n")
print(result["draft"])



Final Draft:

Retrieval-Augmented Generation (RAG) is an advanced technique in natural language processing that combines the strengths of retrieval-based and generation-based models to produce more accurate and contextually relevant responses. At its core, RAG aims to bridge the gap between the exhaustive knowledge retrieval systems and the creative generation capabilities of language models.

RAG operates in two primary phases. In the first phase, a retrieval system is employed to search through a vast repository of documents or text data. This retrieval system indexes the data and quickly identifies relevant passages or documents in response to a user's query. The retrieval process ensures that the generated response is grounded in accurate and up-to-date information.

In the second phase, a language model generates the final response. However, unlike traditional generation models that rely solely on their internal knowledge and training data, RAG incorporates the retrieved informat